# Before Usage Tests

Run this notebook after programming or updating the overlay. With it you can check the RFDC status, verify that both DAC BRAM players can be written and read back through their MMIO arrays, and measures basic host-side BRAM write throughput.

## Imports and Overlay Load

Instantiating `OverlayController` configures the RFSoC clocks, downloads the bitstream, binds `dac0` and `dac2`, and disables both DAC players.

In [ ]:
import time
import numpy as np
from firmware import OverlayController

ol = OverlayController("overlays/rfsocawg.bit")
info = ol.info()
info

Version 1.3


{'bitfile': 'overlays/rfsocawg.bit',
 'clocks': {'lmk_freq_mhz': 245.76,
  'lmx_freq_mhz': 491.52,
  'rf_clock_source': 'internal'},
 'rfdc': {'dac0_sampling_rate_gsps': 9.8304,
  'dac0_fabric_freq_mhz': 614.4,
  'dac2_sampling_rate_gsps': 9.8304,
  'dac2_fabric_freq_mhz': 614.4},
 'dac0': {'name': 'dac0',
  'bram_ip': 'hier_dac_play/axi_bram_ctrl_0',
  'bram_base_addr': 2953314304,
  'bram_bytes': 262144,
  'bram_int16_samples': 131072,
  'waveform_length': 0,
  'enabled': False},
 'dac2': {'name': 'dac2',
  'bram_ip': 'hier_dac2_play/axi_bram_ctrl_0',
  'bram_base_addr': 2953576448,
  'bram_bytes': 262144,
  'bram_int16_samples': 131072,
  'waveform_length': 0,
  'enabled': False}}

In [ ]:
def streamer_enable(player, enabled):
    if enabled:
        player.enable()
    else:
        player.disable()

BUF_LEN = int(info["dac0"]["bram_int16_samples"])
DAC0_SR = float(info["rfdc"]["dac0_sampling_rate_gsps"]) * 1e9
DAC2_SR = float(info["rfdc"]["dac2_sampling_rate_gsps"]) * 1e9

BUF_LEN, DAC0_SR, DAC2_SR

(131072, 9830400000.0, 9830400000.0)

## RFDC Diagnostics

## BRAM MMIO Write-Speed Test

This benchmark measures the PS-to-BRAM write path. It pre-generates one full-BRAM source array, disables the streamer, writes exactly `BUF_LEN` samples repeatedly, and reports effective MiB/s. It does not include waveform generation or readback time.

In [ ]:
def bram_write_speed_test(player, target_bytes=64 * 1024 * 1024):
    player.disable()

    data = np.arange(player.capacity, dtype=np.int16)
    trials = max(10, int(target_bytes / data.nbytes))

    player.buffer[:] = data  # warm-up

    t0 = time.perf_counter()
    for _ in range(trials):
        player.buffer[:] = data
    elapsed = time.perf_counter() - t0

    bytes_written = data.nbytes * trials
    mib_s = bytes_written / elapsed / 1024**2
    result = (player.name, player.capacity, data.nbytes, trials, elapsed, mib_s)
    print(
        f"{player.name}: {player.capacity:8d} samples, {data.nbytes:8d} bytes/write, "
        f"{trials:6d} writes, {mib_s:8.2f} MiB/s"
    )

    return result


bram_write_speed_results = [
    bram_write_speed_test(ol.dac0),
    bram_write_speed_test(ol.dac2),
]

In [ ]:
def print_rfdc_summary():
    rfdc = ol.xrfdc
    print("IPStatus:", rfdc.IPStatus)

    active_dac_tiles = [0, 2]

    for tile_id in active_dac_tiles:
        tile = rfdc.dac_tiles[tile_id]
        print(f"DAC tile {tile_id}: PLLLockStatus={tile.PLLLockStatus}, FIFOStatus={tile.FIFOStatus}")

        block_id = 0
        block = tile.blocks[block_id]
        st = block.BlockStatus
        print(
            f"  block {block_id}: SamplingFreq={st.get('SamplingFreq')}, "
            f"DigitalPathEnabled={st.get('DigitalPathEnabled')}, "
            f"DataPathClocksStatus={st.get('DataPathClocksStatus')}"
        )
        
print_rfdc_summary()

IPStatus: {'DACTileStatus': [{'IsEnabled': 1, 'TileState': 15, 'BlockStatusMask': 1, 'PowerUpState': 1, 'PLLState': 1}, {'IsEnabled': 0, 'TileState': 0, 'BlockStatusMask': 0, 'PowerUpState': 0, 'PLLState': 0}, {'IsEnabled': 1, 'TileState': 15, 'BlockStatusMask': 1, 'PowerUpState': 1, 'PLLState': 1}, {'IsEnabled': 0, 'TileState': 0, 'BlockStatusMask': 0, 'PowerUpState': 0, 'PLLState': 0}], 'ADCTileStatus': [{'IsEnabled': 0, 'TileState': 0, 'BlockStatusMask': 0, 'PowerUpState': 0, 'PLLState': 0}, {'IsEnabled': 0, 'TileState': 0, 'BlockStatusMask': 0, 'PowerUpState': 0, 'PLLState': 0}, {'IsEnabled': 0, 'TileState': 0, 'BlockStatusMask': 0, 'PowerUpState': 0, 'PLLState': 0}, {'IsEnabled': 0, 'TileState': 0, 'BlockStatusMask': 0, 'PowerUpState': 0, 'PLLState': 0}], 'State': 0}
DAC tile 0: PLLLockStatus=2, FIFOStatus=1
  block 0: SamplingFreq=9.8304, DigitalPathEnabled=None, DataPathClocksStatus=1
DAC tile 2: PLLLockStatus=2, FIFOStatus=1
  block 0: SamplingFreq=9.8304, DigitalPathEnabled=No

## BRAM MMIO Readback Test

The test disables the DAC player, writes deterministic `int16` patterns into BRAM through the player buffer, and reads the same memory back. It does not enable RF output.

In [ ]:
def _as_i16_pattern(pattern, n):
    if pattern == "zeros":
        return np.zeros(n, dtype=np.int16)
    if pattern == "ones":
        return np.full(n, 0x7fff, dtype=np.int16)
    if pattern == "alternating":
        x = np.empty(n, dtype=np.int16)
        x[0::2] = np.int16(0x5555)
        x[1::2] = np.int16(-0x5556)  # 0xAAAA as signed int16
        return x
    if pattern == "ramp":
        return (np.arange(n, dtype=np.int32) & 0xffff).astype(np.int16)
    if pattern == "prbs":
        rng = np.random.default_rng(0x12288)
        return rng.integers(-32768, 32767, size=n, dtype=np.int16)
    raise ValueError(pattern)


def bram_array_test(player, n=None, patterns=("zeros", "ones", "alternating", "ramp", "prbs")):
    streamer_enable(player, False)
    n = player.capacity if n is None else min(int(n), player.capacity)
    failures = []
    for pattern in patterns:
        expected = _as_i16_pattern(pattern, n)
        player.buffer[:n] = expected
        time.sleep(0.02)
        got = np.array(player.buffer[:n], dtype=np.int16, copy=True)
        bad = np.flatnonzero(got != expected)
        if len(bad):
            i = int(bad[0])
            failures.append((pattern, len(bad), i, int(expected[i]), int(got[i])))
            print(
                f"FAIL {player.name} {pattern}: mismatches={len(bad)}, first={i}, "
                f"expected={int(expected[i])}, got={int(got[i])}"
            )
        else:
            print(f"PASS {player.name} {pattern}: {n} samples")
    if failures:
        raise AssertionError(f"BRAM array test failed: {failures}")
    return True


test_len = BUF_LEN
bram_array_test(ol.dac0, n=test_len)
bram_array_test(ol.dac2, n=test_len)

PASS dac0 zeros: 131072 samples
PASS dac0 ones: 131072 samples
PASS dac0 alternating: 131072 samples
PASS dac0 ramp: 131072 samples
PASS dac0 prbs: 131072 samples
PASS dac2 zeros: 131072 samples
PASS dac2 ones: 131072 samples
PASS dac2 alternating: 131072 samples
PASS dac2 ramp: 131072 samples
PASS dac2 prbs: 131072 samples


True